# Convolutional Neural Networks

Fully connected networks treat every input independently — an image pixel at the top-left has no special relationship to its neighbor. But images have **spatial structure**: nearby pixels form edges, textures, and objects. **Convolutional Neural Networks (CNNs)** exploit this structure through local connections, shared weights, and pooling — learning visual features hierarchically.

CNNs dominate computer vision: image classification, object detection, segmentation, face recognition, medical imaging, and autonomous driving perception. This notebook explains how convolutions work, how CNN architectures are built, and implements the core operations in NumPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. Why Not Fully Connected Networks for Images?

Consider a 224×224 color image. Flattened, that is $224 \times 224 \times 3 = 150{,}528$ input values. A single fully connected layer with 1,000 neurons requires $150{,}528 \times 1{,}000 \approx 150$ million parameters — just for one layer.

Problems with this approach:

- **Parameter explosion** — far too many weights to learn and store.
- **No spatial awareness** — pixel 1 and pixel 2 are treated the same as pixel 1 and pixel 50,000.
- **Not translation invariant** — a cat in the top-left corner is a completely different input pattern from a cat in the bottom-right.

CNNs solve all three problems with three key ideas: **local connectivity**, **weight sharing**, and **translation equivariance**.

---

## 2. The Convolution Operation

A **convolution** slides a small matrix called a **filter** (or **kernel**) across the input, computing dot products at each position. Each filter detects a specific local pattern.

For a 2D input $X$ and filter $K$ of size $f \times f$:

$$(X * K)_{i,j} = \sum_{m=0}^{f-1} \sum_{n=0}^{f-1} X_{i+m,\, j+n} \cdot K_{m,n}$$

The filter moves across the image (left to right, top to bottom), producing a single **feature map** — a 2D array showing where the pattern was detected.

**Multiple filters** produce multiple feature maps. If a convolutional layer has 32 filters, it outputs 32 feature maps — each detecting a different pattern (edges, corners, textures).

**Key properties:**
- **Local connectivity** — each neuron connects only to a small region of the input.
- **Weight sharing** — the same filter is applied everywhere. One edge detector works across the entire image.
- **Translation equivariance** — if the input shifts, the output shifts correspondingly.

In [ ]:
def conv2d(image, kernel):
    """2D convolution (valid, no padding, stride=1)."""
    h, w = image.shape
    kh, kw = kernel.shape
    out_h, out_w = h - kh + 1, w - kw + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            output[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return output

# Edge detection filters
digits = load_digits()
image = digits.images[0]   # 8×8 digit

sobel_x = np.array([[-1, 0, 1],
                     [-2, 0, 2],
                     [-1, 0, 1]])
sobel_y = np.array([[-1, -2, -1],
                     [ 0,  0,  0],
                     [ 1,  2,  1]])
blur = np.ones((3, 3)) / 9

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("Original")
axes[1].imshow(conv2d(image, sobel_x), cmap="gray")
axes[1].set_title("Vertical edges (Sobel X)")
axes[2].imshow(conv2d(image, sobel_y), cmap="gray")
axes[2].set_title("Horizontal edges (Sobel Y)")
axes[3].imshow(conv2d(image, blur), cmap="gray")
axes[3].set_title("Blur (avg 3×3)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

---

## 3. Stride, Padding, and Output Size

### 3.1 Stride

**Stride** is how many pixels the filter moves at each step. Stride 1 moves one pixel at a time; stride 2 skips every other position, producing a smaller output.

### 3.2 Padding

**Padding** adds pixels around the border of the input (usually zeros). **Same padding** ensures the output has the same spatial dimensions as the input. **Valid padding** means no padding — the output is smaller.

### 3.3 Output Size Formula

For input of size $n \times n$, filter of size $f \times f$, padding $p$, and stride $s$:

$$n_{\text{out}} = \left\lfloor \frac{n + 2p - f}{s} \right\rfloor + 1$$

**Example:** Input $32 \times 32$, filter $3 \times 3$, padding $1$, stride $1$:

$$n_{\text{out}} = \frac{32 + 2 - 3}{1} + 1 = 32$$

Same spatial size preserved.

In [ ]:
def output_size(n, f, p=0, s=1):
    return (n + 2*p - f) // s + 1

configs = [
    (32, 3, 0, 1, "Valid, stride 1"),
    (32, 3, 1, 1, "Same, stride 1"),
    (32, 3, 1, 2, "Same, stride 2"),
    (32, 5, 2, 1, "5×5 filter, pad 2"),
]

print(f"{'Config':25s} {'Output size':>12s}")
print("-" * 40)
for n, f, p, s, label in configs:
    print(f"{label:25s} {output_size(n, f, p, s):12d}")

---

## 4. Pooling Layers

Pooling reduces spatial dimensions, providing **translation invariance** and reducing computation.

### 4.1 Max Pooling

Takes the maximum value in each window. Most common pooling type. Preserves the strongest activation — if an edge was detected anywhere in the window, it survives.

### 4.2 Average Pooling

Takes the average value in each window. Smoother reduction.

Typical setup: $2 \times 2$ window with stride 2 — halves height and width.

Pooling has **no learnable parameters** — it is a fixed downsampling operation.

In [ ]:
def max_pool2d(image, pool_size=2, stride=2):
    h, w = image.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            window = image[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.max(window)
    return output

edges = conv2d(image, sobel_x)
pooled = max_pool2d(edges)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].imshow(image, cmap="gray")
axes[0].set_title(f"Original {image.shape}")
axes[1].imshow(edges, cmap="gray")
axes[1].set_title(f"After conv {edges.shape}")
axes[2].imshow(pooled, cmap="gray")
axes[2].set_title(f"After 2×2 max pool {pooled.shape}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

---

## 5. A Complete CNN Architecture

A typical CNN stacks convolutional and pooling layers, then connects to fully connected layers for classification:

```
Input Image (H × W × C)
    ↓
Conv Layer 1 → ReLU → (feature maps)
    ↓
Max Pool 2×2
    ↓
Conv Layer 2 → ReLU → (more feature maps)
    ↓
Max Pool 2×2
    ↓
Flatten
    ↓
Fully Connected → ReLU
    ↓
Output (softmax for classification)
```

**Channel depth** increases as spatial size decreases: $3 \to 32 \to 64 \to 128$ channels is a common pattern. Early layers detect simple features; deeper layers detect complex combinations.

---

## 6. Parameter Efficiency

CNNs share weights across spatial positions. A $3 \times 3$ filter with 64 input channels and 128 output channels:

$$\text{params} = f \times f \times C_{\text{in}} \times C_{\text{out}} + C_{\text{out}} = 3 \times 3 \times 64 \times 128 + 128 = 73{,}856$$

The same fully connected layer on a $32 \times 32 \times 64$ feature map:

$$\text{params} = 32 \times 32 \times 64 \times 128 + 128 \approx 8{,}400{,}000$$

CNNs are **100× more parameter-efficient** for spatial data because each filter is applied everywhere, not connected to every input.

In [ ]:
def conv_params(f, c_in, c_out):
    return f * f * c_in * c_out + c_out

def fc_params(n_in, n_out):
    return n_in * n_out + n_out

f, c_in, c_out, spatial = 3, 64, 128, 32
conv_p = conv_params(f, c_in, c_out)
fc_p = fc_params(spatial * spatial * c_in, c_out)

print(f"Conv layer ({f}×{f}, {c_in}→{c_out} channels): {conv_p:,} params")
print(f"FC layer  ({spatial*spatial*c_in}→{c_out}):       {fc_p:,} params")
print(f"Conv is {fc_p / conv_p:.0f}× more parameter-efficient")

---

## 7. Hierarchical Feature Learning

CNNs learn features in a hierarchy, similar to the visual cortex:

| Layer | Features Learned | Example |
|-------|-----------------|---------|
| Layer 1 | Edges, colors, gradients | Horizontal/vertical lines |
| Layer 2 | Textures, simple shapes | Circles, corners, stripes |
| Layer 3 | Object parts | Eyes, wheels, handles |
| Layer 4+ | Whole objects | Faces, cars, animals |

Early layers are similar across different CNNs and datasets — edge detectors are universal. Later layers are task-specific. This hierarchy is **learned automatically** from data, not designed by humans.

In [ ]:
# Visualize learned vs hand-crafted filters
filters = {
    "Horizontal edge": np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]]),
    "Vertical edge":   np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]]),
    "Sharpen":         np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
    "Gaussian blur":   np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16,
}

fig, axes = plt.subplots(2, 4, figsize=(14, 6))

for i, (name, filt) in enumerate(filters.items()):
    axes[0, i].imshow(filt, cmap="RdBu", vmin=-2, vmax=2)
    axes[0, i].set_title(name, fontsize=9)
    axes[0, i].axis("off")
    axes[1, i].imshow(conv2d(image, filt), cmap="gray")
    axes[1, i].set_title("Applied to digit", fontsize=9)
    axes[1, i].axis("off")

plt.suptitle("Hand-crafted filters ≈ what CNNs learn in early layers", y=1.02)
plt.tight_layout()
plt.show()

---

## 8. Landmark CNN Architectures

### LeNet-5 (1998)
Yann LeCun's architecture for digit recognition. Conv → pool → conv → pool → FC. Proved CNNs work for vision.

### AlexNet (2012)
Won ImageNet by a large margin. 8 layers, ReLU, dropout, GPU training. Sparked the deep learning revolution.

### VGG (2014)
Showed that depth matters — 16–19 layers of $3 \times 3$ convolutions. Simple, uniform architecture.

### ResNet (2015)
152 layers with **skip connections** (residual blocks). Solved vanishing gradients in very deep networks. Skip connections: $\mathbf{a}^{[l+2]} = f(\mathbf{a}^{[l]}) + F(\mathbf{a}^{[l]})$.

### Modern architectures
EfficientNet, ConvNeXt, Vision Transformer (ViT) — CNNs remain competitive, often combined with attention mechanisms.

---

## 9. Implementing a Simple CNN Pipeline

We build a simplified CNN using our NumPy conv and pool operations, then classify digits.

In [ ]:
def relu(x):
    return np.maximum(0, x)

def softmax(z):
    exp_z = np.exp(z - np.max(z))
    return exp_z / exp_z.sum()

class SimpleCNN:
    """2-layer conv + FC classifier for 8×8 digits."""

    def __init__(self, n_classes=10):
        self.n_classes = n_classes
        # 4 random 3×3 filters
        self.filters = np.random.randn(4, 3, 3) * 0.1
        # After conv(8→6) + pool(6→3): 4 channels × 3 × 3 = 36 features
        self.W_fc = np.random.randn(n_classes, 36) * 0.01
        self.b_fc = np.zeros(n_classes)

    def extract_features(self, image):
        features = []
        for f in self.filters:
            conv_out = relu(conv2d(image, f))
            pooled = max_pool2d(conv_out, pool_size=2, stride=2)
            features.append(pooled.flatten())
        return np.concatenate(features)

    def forward(self, image):
        feat = self.extract_features(image)
        logits = self.W_fc @ feat + self.b_fc
        return softmax(logits), feat

    def predict(self, image):
        probs, _ = self.forward(image)
        return np.argmax(probs)


# Extract features for all digits
digits = load_digits()
cnn = SimpleCNN()

X_features = np.array([cnn.extract_features(img) for img in digits.images])
y = digits.target

X_tr, X_te, y_tr, y_te = train_test_split(X_features, y, test_size=0.2, random_state=42)

# Train FC layer on CNN features (filters are random — features still carry structure)
clf = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42)
clf.fit(X_tr, y_tr)
acc = accuracy_score(y_te, clf.predict(X_te))

print(f"CNN features shape: {X_features.shape}")
print(f"Classifier accuracy on CNN features: {acc:.4f}")
print(f"(Random filters — trained conv filters would perform much better)")

In [ ]:
# Compare: raw pixels vs CNN features
X_pixels = digits.images.reshape(len(digits.images), -1)
X_tr_p, X_te_p, _, _ = train_test_split(X_pixels, y, test_size=0.2, random_state=42)

clf_pixels = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42)
clf_pixels.fit(X_tr_p, y_tr)
acc_pixels = accuracy_score(y_te, clf_pixels.predict(X_te_p))

print(f"Raw pixels (64 features):  accuracy = {acc_pixels:.4f}")
print(f"CNN features (36 features): accuracy = {acc:.4f}")
print("\nCNN features are lower-dimensional yet capture spatial structure.")

---

## 10. Data Augmentation

CNNs need large datasets. **Data augmentation** artificially expands training data by applying random transformations:

- Random horizontal flip
- Random rotation (±15°)
- Random crop and resize
- Color jitter (brightness, contrast, saturation)
- Random zoom

Augmentation acts as regularization — the model sees slightly different versions of each image every epoch, reducing overfitting and improving generalization. It encodes prior knowledge: a cat is still a cat if flipped or rotated.

In [ ]:
# Simulated augmentation on a digit
from scipy.ndimage import rotate, shift

img = digits.images[5]
augmented = [
    ("Original", img),
    ("Rotated 15°", rotate(img, 15, reshape=False, mode="nearest")),
    ("Rotated -20°", rotate(img, -20, reshape=False, mode="nearest")),
    ("Shifted", shift(img, (1, -1), mode="nearest")),
]

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, (title, aug_img) in zip(axes, augmented):
    ax.imshow(aug_img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.suptitle("Data augmentation: same digit, different views", y=1.05)
plt.tight_layout()
plt.show()

---

## 11. 1×1 Convolutions and Depth

A **1×1 convolution** operates on a single spatial position across all channels. It does not mix spatial information — it mixes channels. Used to:

- Change channel depth (dimensionality reduction or expansion).
- Add nonlinearity between layers without spatial mixing.
- Build efficient architectures (Network-in-Network, Inception modules).

Modern CNNs also use **dilated convolutions** (expanded receptive field without pooling) and **depthwise separable convolutions** (MobileNet — factorizes standard conv into depthwise + pointwise for efficiency).

---

## 12. When to Use CNNs

**Use CNNs when:**
- Input has spatial or temporal structure (images, video, audio spectrograms).
- Local patterns matter — nearby elements are related.
- Translation invariance is desired — a feature can appear anywhere.

**Consider alternatives when:**
- Input is tabular — gradient boosting or MLPs are better.
- Input is sequential text — Transformers dominate.
- Very small datasets — transfer learning with pre-trained CNNs is essential.

For image tasks, the standard pipeline is: **pre-trained CNN (ResNet, EfficientNet) + fine-tuning** on your specific dataset.

---

## 13. Summary

**Convolutional Neural Networks** exploit spatial structure through local filters, weight sharing, and pooling. Convolutions detect local patterns; stacking layers builds hierarchical features from edges to objects. Pooling provides downsampling and translation invariance.

CNNs are dramatically more parameter-efficient than fully connected networks for image data. Landmark architectures — LeNet, AlexNet, VGG, ResNet — pushed depth and performance forward. Data augmentation and transfer learning make CNNs practical even with limited data.

CNNs remain the foundation of computer vision, increasingly combined with attention mechanisms and transformer blocks in modern architectures.